# Implement FlashAttention-2 in Triton — Solution

**Difficulty**: 🟣 Expert

**Companies**: NVIDIA, Meta, Together AI, xAI

---

### Problem Statement

Standard attention computes `softmax(Q @ K^T / sqrt(d)) @ V`, which requires **materializing the full N x N attention matrix**. For sequence length N=8192 and batch*heads=32, this matrix alone requires ~2GB of memory in float32. This makes standard attention **O(N^2) in memory**.

**FlashAttention** (Dao et al., 2022) and its successor **FlashAttention-2** solve this by:
1. **Tiling**: Processing Q, K, V in blocks rather than all at once
2. **Online softmax**: Maintaining running softmax statistics (max and sum) across blocks
3. **Never materializing the full N x N matrix**: Peak memory is O(N) instead of O(N^2)

The algorithm processes tiles of Q against ALL tiles of K and V, accumulating the attention output using the online softmax trick to correctly combine partial results from different K,V blocks.

Your task:
1. Implement FlashAttention-2 in **pure PyTorch** (the tiled algorithm with online softmax)
2. Provide the Triton kernel version (with note that it requires GPU)
3. Show that the output matches standard attention exactly
4. Demonstrate the memory advantage

---

### Requirements

1. **Tiled Processing** — Split Q into blocks of `block_size_q` rows, and K,V into blocks of `block_size_kv` rows.
2. **Online Softmax** — For each Q block, iterate over all K,V blocks, maintaining `running_max` and `running_sum` to correctly combine softmax across K blocks.
3. **No Full Matrix** — Never create an N x N tensor. The largest intermediate should be `block_size_q x block_size_kv`.
4. **Correctness** — Output must match `torch.softmax(Q @ K^T / sqrt(d), dim=-1) @ V` within float32 tolerance.

---

### Constraints

- ✅ Pure PyTorch implementation must work on CPU
- ✅ Triton kernel is optional (requires CUDA)
- ✅ Must work for any sequence length (not just multiples of block_size)
- ❌ Do **not** materialize the full N x N attention matrix in the flash implementation

---

<details>
  <summary>💡 Hint</summary>

  **Online softmax across blocks:**
  
  When processing Q_block against K_block_j:
  ```
  S_j = Q_block @ K_block_j^T / sqrt(d)           # (block_q, block_kv)
  block_max_j = S_j.max(dim=-1, keepdim=True)      # max within this block
  new_max = max(running_max, block_max_j)           # global running max
  
  # Correction factor for previously accumulated results
  correction = exp(running_max - new_max)
  
  # Update running sum and output
  P_j = exp(S_j - new_max)                         # local attention weights
  running_sum = running_sum * correction + P_j.sum(dim=-1, keepdim=True)
  running_output = running_output * correction + P_j @ V_block_j
  running_max = new_max
  ```
  
  After all K,V blocks: `output = running_output / running_sum`

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math

TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print("Triton is available!")
except ImportError:
    print("Triton not available. Will use pure PyTorch implementation only.")

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")

In [ ]:
# Test data
torch.manual_seed(42)

# Dimensions
batch_size = 2
n_heads = 4
seq_len = 128
head_dim = 64

# Create Q, K, V tensors: (batch, heads, seq_len, head_dim)
Q = torch.randn(batch_size, n_heads, seq_len, head_dim)
K = torch.randn(batch_size, n_heads, seq_len, head_dim)
V = torch.randn(batch_size, n_heads, seq_len, head_dim)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"\nFull attention matrix would be: {batch_size} x {n_heads} x {seq_len} x {seq_len}")
print(f"= {batch_size * n_heads * seq_len * seq_len * 4 / 1024:.1f} KB")

In [ ]:
def standard_attention(Q, K, V):
    """
    Standard attention: O(N^2) memory.
    This is the reference implementation.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)  # (B, H, N, N) - full matrix!
    attn_weights = torch.softmax(scores, dim=-1)       # (B, H, N, N)
    output = attn_weights @ V                          # (B, H, N, d)
    return output

In [ ]:
def flash_attention_pytorch(Q, K, V, block_size=32):
    """
    FlashAttention-2 implemented in pure PyTorch.
    
    Processes attention in tiles, never materializing the full N x N matrix.
    Uses online softmax to correctly combine results across K,V blocks.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        block_size: tile size for both Q and K,V blocks
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    
    # Output accumulator
    output = torch.zeros_like(Q)
    
    # Number of blocks
    n_blocks = math.ceil(N / block_size)
    
    # Process each Q block
    for q_block_idx in range(n_blocks):
        q_start = q_block_idx * block_size
        q_end = min(q_start + block_size, N)
        q_len = q_end - q_start
        Q_block = Q[:, :, q_start:q_end, :]  # (B, H, block_q, d)
        
        # Initialize online softmax accumulators
        running_max = torch.full((B, H, q_len, 1), float('-inf'), dtype=Q.dtype)
        running_sum = torch.zeros((B, H, q_len, 1), dtype=Q.dtype)
        running_output = torch.zeros((B, H, q_len, d), dtype=Q.dtype)
        
        # Iterate over all K,V blocks
        for kv_block_idx in range(n_blocks):
            kv_start = kv_block_idx * block_size
            kv_end = min(kv_start + block_size, N)
            K_block = K[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)
            V_block = V[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)
            
            # Compute local attention scores: (B, H, block_q, block_kv)
            # This is the ONLY matrix we materialize - much smaller than N x N
            S = Q_block @ K_block.transpose(-2, -1) * scale
            
            # Online softmax update
            # 1. Find max within this block of scores
            block_max = S.max(dim=-1, keepdim=True).values  # (B, H, block_q, 1)
            
            # 2. Compute new global running max
            new_max = torch.maximum(running_max, block_max)
            
            # 3. Correction factor for previously accumulated results
            # When the max changes, previous exp values were too large/small
            correction = torch.exp(running_max - new_max)  # (B, H, block_q, 1)
            
            # 4. Compute local (unnormalized) attention weights
            P = torch.exp(S - new_max)  # (B, H, block_q, block_kv)
            
            # 5. Update running sum: correct old sum + add new weights
            running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
            
            # 6. Update running output: correct old output + add new contribution
            running_output = running_output * correction + P @ V_block
            
            # 7. Update running max
            running_max = new_max
        
        # Final normalization: divide accumulated output by total sum
        output[:, :, q_start:q_end, :] = running_output / running_sum
    
    return output

In [ ]:
# Triton kernel reference (requires GPU to actually run)
# This shows what the GPU kernel would look like

if TRITON_AVAILABLE:
    @triton.jit
    def flash_attention_kernel(
        Q_ptr, K_ptr, V_ptr, O_ptr,
        stride_qb, stride_qh, stride_qn, stride_qd,
        stride_kb, stride_kh, stride_kn, stride_kd,
        stride_vb, stride_vh, stride_vn, stride_vd,
        stride_ob, stride_oh, stride_on, stride_od,
        N, D: tl.constexpr,
        BLOCK_Q: tl.constexpr, BLOCK_KV: tl.constexpr,
    ):
        """
        FlashAttention-2 Triton kernel.
        Each program processes one (batch, head, q_block) tile.
        """
        # Program IDs
        q_block_idx = tl.program_id(0)
        bh_idx = tl.program_id(1)  # combined batch*head index
        
        scale = 1.0 / tl.sqrt(float(D))
        
        # Offsets for this Q block
        q_offset = q_block_idx * BLOCK_Q
        q_range = q_offset + tl.arange(0, BLOCK_Q)
        d_range = tl.arange(0, D)
        q_mask = q_range[:, None] < N
        
        # Load Q block into SRAM
        q_ptrs = Q_ptr + bh_idx * stride_qh + q_range[:, None] * stride_qn + d_range[None, :] * stride_qd
        Q_block = tl.load(q_ptrs, mask=q_mask, other=0.0)
        
        # Initialize accumulators
        running_max = tl.full([BLOCK_Q, 1], float('-inf'), dtype=tl.float32)
        running_sum = tl.zeros([BLOCK_Q, 1], dtype=tl.float32)
        running_out = tl.zeros([BLOCK_Q, D], dtype=tl.float32)
        
        # Loop over K,V blocks
        n_kv_blocks = tl.cdiv(N, BLOCK_KV)
        for kv_idx in range(n_kv_blocks):
            kv_offset = kv_idx * BLOCK_KV
            kv_range = kv_offset + tl.arange(0, BLOCK_KV)
            kv_mask = kv_range[None, :] < N
            
            # Load K block
            k_ptrs = K_ptr + bh_idx * stride_kh + kv_range[None, :] * stride_kn + d_range[:, None] * stride_kd
            K_block = tl.load(k_ptrs, mask=kv_mask, other=0.0)  # (D, BLOCK_KV)
            
            # Compute scores: Q @ K^T
            S = tl.dot(Q_block, K_block) * scale  # (BLOCK_Q, BLOCK_KV)
            
            # Online softmax
            block_max = tl.max(S, axis=1)[:, None]
            new_max = tl.maximum(running_max, block_max)
            correction = tl.exp(running_max - new_max)
            P = tl.exp(S - new_max)
            running_sum = running_sum * correction + tl.sum(P, axis=1)[:, None]
            
            # Load V block and accumulate
            v_ptrs = V_ptr + bh_idx * stride_vh + kv_range[:, None] * stride_vn + d_range[None, :] * stride_vd
            v_mask = kv_range[:, None] < N
            V_block = tl.load(v_ptrs, mask=v_mask, other=0.0)
            running_out = running_out * correction + tl.dot(P.to(V_block.dtype), V_block)
            running_max = new_max
        
        # Normalize and store
        result = running_out / running_sum
        o_ptrs = O_ptr + bh_idx * stride_oh + q_range[:, None] * stride_on + d_range[None, :] * stride_od
        tl.store(o_ptrs, result, mask=q_mask)
    
    print("Triton flash attention kernel defined.")
else:
    print("Triton not available. The kernel code above shows the GPU implementation.")
    print("The pure PyTorch flash_attention_pytorch is the main implementation to study.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING FLASH ATTENTION")
print("=" * 60)

# Reference output
ref_output = standard_attention(Q, K, V)

# Flash attention output
flash_output = flash_attention_pytorch(Q, K, V, block_size=32)

# Test 1: Correctness
print("\n--- Correctness Test ---")
max_err = (flash_output - ref_output).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")
assert torch.allclose(flash_output, ref_output, atol=1e-5), "Flash attention output MISMATCH!"
print("PASSED")

# Test 2: Different block sizes
print("\n--- Block Size Robustness Test ---")
for bs in [16, 32, 64]:
    flash_out = flash_attention_pytorch(Q, K, V, block_size=bs)
    err = (flash_out - ref_output).abs().max().item()
    assert torch.allclose(flash_out, ref_output, atol=1e-5), f"Failed for block_size={bs}"
    print(f"  block_size={bs}: max_err={err:.2e} PASSED")

# Test 3: Non-divisible sequence length
print("\n--- Non-Divisible Sequence Length Test ---")
Q2 = torch.randn(1, 1, 50, 32)
K2 = torch.randn(1, 1, 50, 32)
V2 = torch.randn(1, 1, 50, 32)
ref2 = standard_attention(Q2, K2, V2)
flash2 = flash_attention_pytorch(Q2, K2, V2, block_size=16)
err2 = (flash2 - ref2).abs().max().item()
assert torch.allclose(flash2, ref2, atol=1e-5), "Non-divisible seq len FAILED!"
print(f"  seq_len=50, block_size=16: max_err={err2:.2e} PASSED")

# Test 4: Memory comparison
print("\n--- Memory Analysis ---")
N = seq_len
d = head_dim
bs = 32
standard_mem = N * N  # full attention matrix
flash_mem_actual = bs * bs  # actual peak: one tile of scores
print(f"  Standard attention peak: {N}x{N} = {standard_mem} elements")
print(f"  Flash attention peak:    {bs}x{bs} = {flash_mem_actual} elements per tile")
print(f"  Memory reduction: {standard_mem / flash_mem_actual:.1f}x")

# For longer sequences, the advantage is even more dramatic
print("\n--- Scaling Analysis ---")
for N_test in [256, 1024, 4096, 8192]:
    standard = N_test * N_test * 4  # bytes in float32
    flash = bs * bs * 4
    print(f"  N={N_test:>5d}: standard={standard/1024/1024:>8.1f} MB, flash_tile={flash/1024:.1f} KB, ratio={standard // flash}x")

print("\nAll tests passed!")